In [2]:
import os
import io
import pandas as pd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

# ----------------------------
# 1. Miljøvariabler
# ----------------------------
load_dotenv(".env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "processeddata"

# ----------------------------
# 2. LES FORBRUK
# ----------------------------
blob_client1 = blob_service_client.get_blob_client(
    container=container_name,
    blob="FRIKSTAD_processed.csv"
)
data1 = blob_client1.download_blob().readall()
df_consumption = pd.read_csv(io.BytesIO(data1))

# ----------------------------
# 3. LES VÆR
# ----------------------------
blob_client2 = blob_service_client.get_blob_client(
    container=container_name,
    blob="FRIKSTAD_weather.csv"
)
data2 = blob_client2.download_blob().readall()
df_weather = pd.read_csv(io.BytesIO(data2))

print("Begge filer lastet!")

# ----------------------------
# 1. Fix timestamp
# ----------------------------
df_consumption['timestamp'] = pd.to_datetime(df_consumption['timestamp'])
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])


# ----------------------------
# 2. Merge forbruk + vær
# ----------------------------
df = pd.merge(df_consumption, df_weather, on="timestamp", how="inner")

print("Antall rader etter merge:", len(df))

# ----------------------------
# 3. Split før / etter
# ----------------------------
df_before = df[df["norgespris"] == 0]
df_after = df[df["norgespris"] == 1]

print("Før:", len(df_before))
print("Etter:", len(df_after))

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ----------------------------
# Velg features
# ----------------------------
feature_cols = [
    "hour",
    "weekday",
    "month",
    "is_weekend",
    "is_holiday",
    "air_temperature",
    "wind_speed",
    "precipitation_mm"
]

target_col = "value_kwh"

# ============================
#  FØR NORGESPRIS
# ============================
X_before = df_before[feature_cols]
y_before = df_before[target_col]

model_before = LinearRegression()
model_before.fit(X_before, y_before)

pred_before = model_before.predict(X_before)

import numpy as np

print("\n--- FØR NORGESPRIS ---")
print("R2:", r2_score(y_before, pred_before))
print("RMSE:", np.sqrt(mean_squared_error(y_before, pred_before)))

print("\nCoefficients:")
for col, coef in zip(feature_cols, model_before.coef_):
    print(f"{col}: {coef}")

# ============================
#  ETTER NORGESPRIS
# ============================
X_after = df_after[feature_cols]
y_after = df_after[target_col]

model_after = LinearRegression()
model_after.fit(X_after, y_after)

pred_after = model_after.predict(X_after)

print("\n--- ETTER NORGESPRIS ---")
print("R2:", r2_score(y_after, pred_after))
print("RMSE:", np.sqrt(mean_squared_error(y_after, pred_after)))

print("\nCoefficients:")
for col, coef in zip(feature_cols, model_after.coef_):
    print(col, ":", coef)

Begge filer lastet!
Antall rader etter merge: 24728358
Før: 13185986
Etter: 11542372

--- FØR NORGESPRIS ---
R2: 0.021262222474540904
RMSE: 2.215522048897995

Coefficients:
hour: 0.022669414489560002
weekday: 0.014742647267002378
month: -0.011314275668901955
is_weekend: 0.0030263499426465904
is_holiday: 0.12964966110395895
air_temperature: -0.0660429284489539
wind_speed: 0.02685042748467417
precipitation_mm: 0.029493731049973233

--- ETTER NORGESPRIS ---
R2: 0.04119449705985778
RMSE: 2.3136705526184556


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).